# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and analyze the FAIR^2 dataset (colorectal cancer survivors) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a [Croissant schema](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant JSON-LD schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata (this does not fetch data files yet)
dataset = mlc.Dataset(croissant_url)

print(f"Dataset: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}\n")
print(f"Citation: {dataset.metadata.cite_as}\n")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

We'll print all record sets, their key fields, and columns, referencing entities strictly by their `@id`s.

In [ ]:
# List all record sets and their fields by @id
print("Available Record Sets (referenced by @id):\n")
record_sets_metadata = dataset.metadata.record_sets
record_set_ids = []
for rs in record_sets_metadata:
    print(f"  RecordSet name: {rs.name}, @id: {rs.id}")
    record_set_ids.append(rs.id)
    if hasattr(rs, 'fields') and rs.fields:
        print("    Fields:")
        for field in rs.fields:
            print(f"      - {field.name} (@id: {field.id}, dataType: {getattr(field, 'data_type', '?')})")
    if hasattr(rs, 'columns') and rs.columns:
        print("    Columns:")
        for column in rs.columns:
            print(f"      - {column.name} (@id: {column.id}, dataType: {getattr(column, 'data_type', '?')})")
    print()
    
print("\nTo preview records from a record set, we reference entities by their @id.\n")
# Optionally, show a preview of records for each record set:
for rset_id in record_set_ids:
    print(f"Sample record from record set @id: {rset_id}")
    for i, rec in enumerate(dataset.records(record_set=rset_id)):
        print(rec)
        if i == 0:
            break
    print()

## 3. Data Extraction
Load data from each record set into a pandas DataFrame for further analysis.

- Record sets and columns are always referenced by their `@id`.
- DataFrame keys and selections use these `@id`s to maintain consistency.

Let's load all record sets into DataFrames indexed by record set `@id`.

In [ ]:
# Extract and store each record set by @id as a DataFrame
dfs = {}
for rset_id in record_set_ids:
    print(f"Loading records for record set @id: {rset_id}")
    records = list(dataset.records(record_set=rset_id))
    if records:
        dfs[rset_id] = pd.DataFrame(records)
        print(f"  Columns: {dfs[rset_id].columns.tolist()}")
        display(dfs[rset_id].head(2))
    else:
        print("  No records found.")
print("\nAll record sets loaded into the `dfs` dictionary, keyed by record set @id.")

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps:
- Filter records by criteria (e.g., values of a numeric field)
- Normalize a numeric column (standard score)
- Group by a categorical field and aggregate

All columns and fields are referenced via their `@id` (as per the Croissant specification).

Below, we will illustrate this with one of the available record sets. We will detect a likely numeric field for demonstration; please adapt as appropriate for your use case.

In [ ]:
# Choose a record set for demonstration. Select the first one with records.
primary_rs_id = None
for rset_id in dfs:
    if not dfs[rset_id].empty:
        primary_rs_id = rset_id
        break
if primary_rs_id is None:
    raise ValueError('No record set with data!')

print(f"Using record set @id: {primary_rs_id}\n")

df = dfs[primary_rs_id].copy()
# Try to infer a likely numeric field by dtype. Fallback to the first field.
numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
if numeric_fields:
    numeric_field_id = numeric_fields[0]
else:
    numeric_field_id = df.columns[0]  # As fallback
print(f"Numeric field selected for EDA: {numeric_field_id}")

# Set a threshold for illustration (pick 10, or lower if max is small):
thresh = 10
if df[numeric_field_id].max() < 10:
    thresh = df[numeric_field_id].max() * 0.7

filtered_df = df[df[numeric_field_id] > thresh]
print(f"Filtered records with {numeric_field_id} > {thresh} (total: {len(filtered_df)}):\n")
display(filtered_df.head())

filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Attempt to group by a non-numeric field
non_numeric = [col for col in df.columns if not pd.api.types.is_numeric_dtype(df[col]) and col != numeric_field_id]
if non_numeric:
    group_field_id = non_numeric[0]
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame(f"mean_{numeric_field_id}")
    print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
    display(grouped_df.head())
else:
    print("No non-numeric field found for grouping.")

## 5. Visualization
Visualize a distribution or relationship using the extracted DataFrame and column `@id`s.

- We'll show a histogram for the numeric field and a boxplot if a group field was found.
- Axis labels use the entity `@id`s to remain schema-compliant.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram for the numeric field
plt.figure(figsize=(7,4))
sns.histplot(df[numeric_field_id].dropna(), bins=10, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# Boxplot for group vs. numeric (if exists)
if non_numeric:
    plt.figure(figsize=(10,4))
    sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
This notebook demonstrated step-by-step exploration and analysis of a Croissant-compliant FAIR^2 dataset using `mlcroissant`, referencing all dataset entities by their `@id`.

- Record sets, fields, and columns were managed via their unique `@id`s per the Croissant specification.
- Data loading, EDA, and basic visualization were performed using standard Python data science tools.

You can now extend this workflow for other FAIR/Open datasets with Croissant schemas for reproducible ML data science!